# XFakeNet – Week 1 on Google Colab

Runs the full Week 1 pipeline on the **whole dataset**: data prep, three-class label mapping (Fabricated / Misleading / Credible),
baseline RoBERTa, FakeNewsNet image-pair count, and (optional) TextFooler.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. `Runtime → Run all`
3. Click **Allow** when asked to connect Google Drive (everything is saved there, so a disconnect loses nothing).

At the end, section 8 prints a results table for your mentor and downloads a zip of the results.

## 1. Check GPU and connect Google Drive

In [ ]:
import os, subprocess, sys
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE – go to Runtime > Change runtime type > T4 GPU, then Run all again")

USE_DRIVE = True   # set False to work only in /content (lost when the session ends)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/xfakenet"
else:
    WORK = "/content/xfakenet"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("Working in", WORK)

## 2. Write the project code
(These cells just save the scripts into the project folder.)

In [ ]:
import os
for d in ['configs','src/xfakenet','scripts','docs','logs','data/raw','results']:
    os.makedirs(d, exist_ok=True)
print('folders ready')

In [ ]:
%%writefile configs/week1.yaml
seed: 42
paths:
  raw: data/raw
  processed: data/processed
  images: data/images
  runs: runs

liar:
  dir: data/raw/liar            # expects train.tsv / valid.tsv / test.tsv

extra_classes:
  enabled: false      # three-class setup: LIAR only
  source: tshp17
  file: data/raw/tshp17/fulltrain.csv   # columns: label,text (no header)
  test_file: data/raw/tshp17/balancedtest.csv
  use_hoax_trusted: false
  # Articles are long; LIAR statements are ~18 words. Truncate to the first
  # sentences up to this many words so length is not a shortcut for the class.
  max_words: 40
  # Cap per class so Satire/Propaganda stay minority classes (as in the outline).
  max_per_class: 1500

fakenewsnet:
  dir: data/raw/FakeNewsNet/dataset
  sample_size: 1000            # articles to try for images (balanced fake/real)
  min_image_side: 100          # px; smaller images (icons, pixels) are discarded
  workers: 16
  timeout: 10
  use_wayback: true

baseline:
  model: roberta-base
  max_len: 64
  epochs: 4
  batch_size: 32
  lr: 2.0e-5
  warmup_ratio: 0.06
  weight_decay: 0.01
  fp16: true

# ---------------- Week 2 ----------------
augmentation:                  # final training mix (week 3): 70% original / 15% / 15%
  original_share: 70
  textfooler_share: 15
  paraphrase_share: 15

cscore:
  clip_model: openai/clip-vit-base-patch32
  batch_size: 64
  percentile: 95               # alternative threshold: 95th percentile of aligned C-scores (valid)
  out_dir: results/cscore

paraphrase:
  models: [gemini-3.5-flash-lite, gemini-3.8-flash, gemini-3.1-flash-lite, gemini-2.5-flash]
  batch_size: 20               # statements per API call
  temperature: 0.9
  sleep_s: 4                   # pause between calls (free-tier rate limits)
  test_n: 500
  out_train: data/augment/paraphrase_train.jsonl
  out_test: data/augment/paraphrase_test.jsonl

textfooler:
  victim_dir: runs/baseline/best
  embeddings_url: https://github.com/nmrksic/counter-fitting/raw/master/word_vectors/counter-fitted-vectors.txt.zip
  embeddings_path: data/raw/counter-fitted-vectors.txt.zip
  sim_model: sentence-transformers/all-MiniLM-L6-v2
  k: 50                        # synonyms per word (paper: 50)
  sim_threshold: 0.80
  max_change: 0.25             # at most 25% of words replaced
  query_budget: 2000
  test_n: 500
  out_train: data/augment/textfooler_train.jsonl
  out_test: data/augment/textfooler_test.jsonl

model_v1:
  lrs: [1.0e-5, 2.0e-5, 3.0e-5]
  epochs: 5

# ---------------- Week 3 ----------------
xfakenet:                      # RoBERTa + jointly trained token/segment attention
  encoder: roberta-base
  max_len: 64
  max_segments: 8
  att_dim: 256
  epochs: 4
  batch_size: 32
  lr: 2.0e-5                   # encoder
  head_lr: 1.0e-4              # attention layers + classifier
  warmup_ratio: 0.06
  weight_decay: 0.01
  highlight_threshold: 0.65    # outline: tokens above T = 0.65 are highlighted

modality:
  C: 0.1                       # regularisation of the text/image logistic models (389 items)

week3:
  out_dir: results/week3

# ---------------- Week 4 ----------------
week4:
  out_dir: results/week4

deception:
  n_mismatched: 200            # shuffled pairs added to the 389 real ones for labelling
  batch_size: 5                # image-text pairs per Gemini call
  image_side: 512
  sleep_s: 4
  C: 0.5
  min_per_class: 5
  handcheck_n: 100
  labels: data/deception/gemini_labels.jsonl
  features: data/deception/clip_features.npz
  oof: data/deception/classifier_oof.csv
  model: runs/deception_clf.joblib
  handcheck_sample: data/deception/handcheck_sample.csv
  handcheck_results: data/deception/handcheck_results.csv

pipeline:
  modality_model: runs/modality.joblib

explain:                       # faithful explanations (week 4 fix)
  permutations_gpu: 10
  permutations_cpu: 6
  cpu_statements: 400
  chunk: 50

In [ ]:
%%writefile configs/label_mapping.yaml
# XFakeNet five-class label mapping (frozen 2026-10-21, Week 1)
# Every later step (training, augmentation, evaluation) reads this file.
# Change it only with a note in docs/label_mapping.md and a full retrain.

# Three-class setup (changed 20 Oct 2026): Satire/Propaganda dropped because the
# only available source (TSHP-17) taught the model website style, not the class
# (Propaganda F1 0.99 on same-site validation, 0.08 on unseen-site test).
# For the old five-class setup use:
#   classes: [Fabricated, Misleading, Satire, Propaganda, Credible]
#   and set extra_classes.enabled: true in week1.yaml
classes: [Fabricated, Misleading, Credible]   # index order = model label ids

liar:
  # LIAR's six PolitiFact truth ratings -> XFakeNet classes
  # Keys are quoted on purpose: unquoted true/false become YAML booleans.
  "pants-fire":  Fabricated
  "false":       Fabricated
  "barely-true": Misleading
  "half-true":   Misleading
  "mostly-true": Credible
  "true":        Credible

# LIAR has no Satire or Propaganda signal. These two classes come from an
# external labelled source (default: TSHP-17, Rashkin et al. 2017).
tshp17:
  # TSHP-17 integer labels: 1=Satire 2=Hoax 3=Propaganda 4=Trusted
  1: Satire
  3: Propaganda
  # 2 (Hoax) and 4 (Trusted) are NOT used by default: they would add
  # long-article Fabricated/Credible examples whose style differs from LIAR.
  # Set use_hoax_trusted: true in configs/week1.yaml to include them.
  2: Fabricated
  4: Credible

fakenewsnet:
  # Binary only; used for C-score calibration/evaluation, not five-class training.
  "fake": Fabricated
  "real": Credible

In [ ]:
%%writefile src/xfakenet/__init__.py
# package marker

In [ ]:
%%writefile src/xfakenet/common.py
"""Shared helpers: config loading, label mapping, seeding, text cleanup."""
from __future__ import annotations

import json
import random
import re
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[2]


def load_yaml(path: str | Path) -> dict:
    with open(ROOT / path if not Path(path).is_absolute() else path) as f:
        return yaml.safe_load(f)


def load_config() -> dict:
    return load_yaml("configs/week1.yaml")


def load_mapping() -> dict:
    return load_yaml("configs/label_mapping.yaml")


def class_names() -> list[str]:
    return load_mapping()["classes"]


def label2id() -> dict[str, int]:
    return {c: i for i, c in enumerate(class_names())}


def p(rel: str | Path) -> Path:
    """Resolve a repo-relative path."""
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def set_seed(seed: int) -> None:
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


_WS = re.compile(r"\s+")
_SENT = re.compile(r"(?<=[.!?])\s+")


def clean_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.replace("“", '"').replace("”", '"').replace("’", "'")
    return _WS.sub(" ", s).strip()


def truncate_words(text: str, max_words: int) -> str:
    """Keep whole leading sentences up to max_words; hard-cut if the first sentence is longer."""
    text = clean_text(text)
    out, n = [], 0
    for sent in _SENT.split(text):
        w = sent.split()
        if n + len(w) > max_words:
            if not out:
                out.append(" ".join(w[:max_words]))
            break
        out.append(sent)
        n += len(w)
    return " ".join(out)


def write_json(obj, path: str | Path) -> None:
    with open(p(path), "w") as f:
        json.dump(obj, f, indent=2, default=str)

In [ ]:
%%writefile scripts/02_prepare_liar.py
"""Load LIAR, clean it, and apply the five-class mapping.

Reads the original TSVs from data/raw/liar/{train,valid,test}.tsv.
(The Hugging Face copy `ucsbnlp/liar` is script-based and no longer loads.)

Output: data/processed/liar_{train,valid,test}.csv with columns
    id, text, liar_label, label, source, split, speaker, party, subject, context
"""
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import clean_text, load_config, load_mapping, p, write_json  # noqa: E402

COLS = ["id", "liar_label", "text", "subject", "speaker", "job", "state", "party",
        "barely_true_ct", "false_ct", "half_true_ct", "mostly_true_ct", "pants_fire_ct", "context"]
SPLITS = {"train": "train.tsv", "valid": "valid.tsv", "test": "test.tsv"}


def load_tsv(d: Path) -> dict[str, pd.DataFrame]:
    out = {}
    for split, fn in SPLITS.items():
        df = pd.read_csv(d / fn, sep="\t", header=None, names=COLS, quoting=3,
                         dtype=str, keep_default_na=False)
        out[split] = df
    return out


def main() -> None:
    cfg, mapping = load_config(), load_mapping()["liar"]
    d = p(cfg["liar"]["dir"])
    if not (d / "train.tsv").exists():
        sys.exit(f"[liar] {d}/train.tsv not found. Download liar_dataset.zip "
                 "(UCSB, or search 'LIAR dataset' on Kaggle) and unzip it into that folder.")
    raw = load_tsv(d)

    stats = {}
    for split, df in raw.items():
        df = df.copy()
        df["liar_label"] = df["liar_label"].str.strip().str.lower()
        unknown = set(df["liar_label"]) - set(mapping)
        if unknown:
            raise ValueError(f"Unmapped LIAR labels in {split}: {unknown}")
        df["text"] = df["text"].map(clean_text)
        n0 = len(df)
        df = df[df["text"].str.len() > 0].drop_duplicates(subset=["text"])
        df["label"] = df["liar_label"].map(mapping)
        df["source"] = "liar"
        df["split"] = split
        keep = ["id", "text", "liar_label", "label", "source", "split",
                "speaker", "party", "subject", "context"]
        df = df[[c for c in keep if c in df.columns]]
        df.to_csv(p(f"{cfg['paths']['processed']}/liar_{split}.csv"), index=False)
        stats[split] = {
            "rows_raw": n0,
            "rows_clean": len(df),
            "liar_labels": df["liar_label"].value_counts().to_dict(),
            "xfakenet_labels": df["label"].value_counts().to_dict(),
            "mean_words": round(df["text"].str.split().str.len().mean(), 1),
        }
        print(f"[liar] {split}: {n0} -> {len(df)} rows | {stats[split]['xfakenet_labels']}")
    write_json(stats, f"{cfg['paths']['processed']}/liar_stats.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/03_prepare_extra_classes.py
"""Build Satire and Propaganda examples from TSHP-17 (Rashkin et al., 2017).

LIAR's six truth ratings contain no satire or propaganda signal, so these two
classes need an external source. TSHP-17 labels news articles as
1=Satire, 2=Hoax, 3=Propaganda, 4=Trusted.

To keep the style close to LIAR's short statements, each article is cut to its
leading sentences (<= max_words) so the model cannot cheat on length.

Input : data/raw/tshp17/fulltrain.csv (+ balancedtest.csv), no header: label,text
Output: data/processed/extra_{train,valid,test}.csv
"""
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, load_mapping, p, truncate_words, write_json  # noqa: E402


def cap_per_class(d: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    parts = [g.sample(min(len(g), n), random_state=seed) for _, g in d.groupby("label")]
    return pd.concat(parts) if parts else d


def read(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=["tshp_label", "text"], usecols=[0, 1], dtype=str)
    # tolerate a header row or stray lines: keep rows whose label is 1-4
    df["tshp_label"] = pd.to_numeric(df["tshp_label"], errors="coerce")
    df = df[df["tshp_label"].isin([1, 2, 3, 4])].copy()
    df["tshp_label"] = df["tshp_label"].astype(int)
    return df


def main() -> None:
    cfg = load_config()
    ec = cfg["extra_classes"]
    out_dir = cfg["paths"]["processed"]
    if not ec["enabled"]:
        print("[extra] disabled in config; skipping")
        return
    f = p(ec["file"])
    if not f.exists():
        print(f"[extra] {f} not found. Satire/Propaganda will be EMPTY until you add a source.\n"
              "        See docs/label_mapping.md, section 'Getting Satire and Propaganda data'.")
        return

    mapping = {int(k): v for k, v in load_mapping()["tshp17"].items()}
    keep = {1, 3} | ({2, 4} if ec["use_hoax_trusted"] else set())

    df = read(f)
    test_f = p(ec["test_file"])
    df_test = read(test_f) if test_f.exists() else None

    def prep(d: pd.DataFrame) -> pd.DataFrame:
        d = d[d["tshp_label"].isin(keep)].copy()
        d["text"] = d["text"].map(lambda t: truncate_words(t, ec["max_words"]))
        d = d[d["text"].str.split().str.len() >= 5].drop_duplicates(subset=["text"])
        d["label"] = d["tshp_label"].map(mapping)
        d["source"] = "tshp17"
        return d

    df = prep(df)
    # cap per class, reproducibly
    df = cap_per_class(df, ec["max_per_class"], cfg["seed"])

    if df_test is not None:
        test = cap_per_class(prep(df_test), max(1, ec["max_per_class"] // 8), cfg["seed"])
        train, valid = train_test_split(df, test_size=0.1, stratify=df["label"], random_state=cfg["seed"])
    else:
        train, rest = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=cfg["seed"])
        valid, test = train_test_split(rest, test_size=0.5, stratify=rest["label"], random_state=cfg["seed"])

    stats = {}
    for split, d in [("train", train), ("valid", valid), ("test", test)]:
        d = d.copy()
        d["split"] = split
        d["id"] = [f"tshp-{split}-{i}" for i in range(len(d))]
        d[["id", "text", "label", "source", "split"]].to_csv(p(f"{out_dir}/extra_{split}.csv"), index=False)
        stats[split] = d["label"].value_counts().to_dict()
        print(f"[extra] {split}: {stats[split]}")
    write_json(stats, f"{out_dir}/extra_stats.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/04_build_corpus.py
"""Merge LIAR and the extra-class source into one five-class corpus.

Output: data/processed/corpus_{train,valid,test}.csv  (columns: id,text,label,label_id,source)
        data/processed/corpus_stats.json  (class counts + class weights for the loss)
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import class_names, label2id, load_config, p, write_json  # noqa: E402


def main() -> None:
    cfg = load_config()
    d = cfg["paths"]["processed"]
    l2i, classes = label2id(), class_names()
    stats = {}
    for split in ["train", "valid", "test"]:
        parts = [pd.read_csv(p(f"{d}/liar_{split}.csv"))]
        extra = p(f"{d}/extra_{split}.csv")
        if cfg["extra_classes"]["enabled"] and extra.exists():
            parts.append(pd.read_csv(extra))
        df = pd.concat(parts, ignore_index=True)[["id", "text", "label", "source"]]
        df = df[df["label"].isin(classes)]   # drop anything outside the active class list
        df = df.drop_duplicates(subset=["text"])
        df["label_id"] = df["label"].map(l2i)
        df = df.sample(frac=1.0, random_state=cfg["seed"]).reset_index(drop=True)
        df.to_csv(p(f"{d}/corpus_{split}.csv"), index=False)
        counts = {c: int((df["label"] == c).sum()) for c in classes}
        stats[split] = {"n": len(df), "counts": counts,
                        "by_source": df.groupby(["source", "label"]).size().unstack(fill_value=0).to_dict()}
        print(f"[corpus] {split}: n={len(df)} {counts}")

    # Inverse-frequency class weights from TRAIN, normalised to mean 1. Absent classes get 0.
    c = np.array([stats["train"]["counts"][k] for k in classes], dtype=float)
    present = c > 0
    w = np.zeros_like(c)
    w[present] = c[present].sum() / (present.sum() * c[present])
    w[present] /= w[present].mean()
    stats["class_weights"] = dict(zip(classes, np.round(w, 4).tolist()))
    missing = [k for k, n in zip(classes, c) if n == 0]
    if missing:
        stats["warning"] = f"No training data for {missing}; model will never predict them."
        print(f"[corpus] WARNING: no training data for {missing}")
    print(f"[corpus] class weights: {stats['class_weights']}")
    write_json(stats, f"{d}/corpus_stats.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/05_fetch_fnn_images.py
"""Download FakeNewsNet article images and count usable headline-image pairs.

For each sampled article we fetch the page (live, then the Wayback Machine if the
live link is dead), take og:image / twitter:image (or the first large <img>),
download it and check it opens and is at least min_image_side px on each side.

Resumable: already-processed ids in the manifest are skipped, so you can
kill and restart it freely.

Input : data/raw/FakeNewsNet/dataset/{politifact,gossipcop}_{fake,real}.csv
Output: data/images/<id>.jpg
        data/processed/fnn_manifest.jsonl     (one row per article, with status)
        data/processed/fnn_pairs.csv          (usable pairs only: id,title,label,image_path,source_url)
        data/processed/fnn_image_summary.json (the counts for the week-1 report)

Usage : python scripts/05_fetch_fnn_images.py [--limit N] [--summary-only]
"""
from __future__ import annotations

import argparse
import io
import json
import sys
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import clean_text, load_config, p, write_json  # noqa: E402

UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) XFakeNet-research/0.1"}
_lock = threading.Lock()


def load_fnn(d: Path) -> pd.DataFrame:
    frames = []
    for site in ["politifact", "gossipcop"]:
        for lab in ["fake", "real"]:
            f = d / f"{site}_{lab}.csv"
            if f.exists():
                df = pd.read_csv(f, dtype=str, keep_default_na=False)
                df["site"], df["fnn_label"] = site, lab
                frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No FakeNewsNet CSVs in {d}. Run scripts/01_download_data.sh first.")
    df = pd.concat(frames, ignore_index=True)
    df["title"] = df["title"].map(clean_text)
    df = df[(df["news_url"].str.len() > 0) & (df["title"].str.len() > 0)]
    return df.drop_duplicates(subset=["id"])


def sample_balanced(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    """Balanced over fake/real and over sites, as far as the data allows."""
    groups = df.groupby(["site", "fnn_label"])
    per = max(1, n // groups.ngroups)
    parts = [g.sample(min(len(g), per), random_state=seed) for _, g in groups]
    out = pd.concat(parts)
    if len(out) < n:  # top up from whatever is left
        rest = df.drop(out.index)
        out = pd.concat([out, rest.sample(min(len(rest), n - len(out)), random_state=seed)])
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def norm_url(u: str) -> str:
    u = u.strip()
    return u if u.startswith(("http://", "https://")) else "http://" + u


def get(url: str, timeout: int) -> requests.Response | None:
    try:
        r = requests.get(url, headers=UA, timeout=timeout, allow_redirects=True)
        return r if r.status_code == 200 else None
    except requests.RequestException:
        return None


def wayback(url: str, timeout: int) -> str | None:
    r = get(f"https://archive.org/wayback/available?url={url}&timestamp=20180101", timeout)
    if r is None:
        return None
    try:
        snap = r.json()["archived_snapshots"]["closest"]
        return snap["url"] if snap.get("available") else None
    except (KeyError, ValueError):
        return None


def find_image_url(html: str, base: str) -> str | None:
    soup = BeautifulSoup(html, "html.parser")
    for attr, key in [("property", "og:image"), ("name", "twitter:image"),
                      ("property", "og:image:url"), ("name", "twitter:image:src")]:
        tag = soup.find("meta", attrs={attr: key})
        if tag and tag.get("content"):
            return urljoin(base, tag["content"].strip())
    for img in soup.find_all("img", src=True):
        src = img["src"]
        if any(s in src.lower() for s in ("logo", "icon", "sprite", "pixel", ".svg", "avatar")):
            continue
        return urljoin(base, src)
    return None


def process(row: dict, cfg: dict) -> dict:
    from PIL import Image  # local import keeps worker startup cheap

    fc = cfg["fakenewsnet"]
    url = norm_url(row["news_url"])
    rec = {"id": row["id"], "title": row["title"], "fnn_label": row["fnn_label"],
           "site": row["site"], "news_url": url, "status": None, "via": None}

    page, via = get(url, fc["timeout"]), "live"
    if page is None and fc["use_wayback"]:
        snap = wayback(url, fc["timeout"])
        page, via = (get(snap, fc["timeout"]), "wayback") if snap else (None, None)
    if page is None:
        rec["status"] = "page_dead"
        return rec
    rec["via"] = via

    img_url = find_image_url(page.text, page.url)
    if not img_url:
        rec["status"] = "no_image_tag"
        return rec
    rec["image_url"] = img_url

    r = get(img_url, fc["timeout"])
    if r is None and via == "live" and fc["use_wayback"]:
        snap = wayback(img_url, fc["timeout"])
        r = get(snap, fc["timeout"]) if snap else None
    if r is None:
        rec["status"] = "image_dead"
        return rec
    try:
        im = Image.open(io.BytesIO(r.content)).convert("RGB")
    except Exception:
        rec["status"] = "image_corrupt"
        return rec
    if min(im.size) < fc["min_image_side"]:
        rec["status"] = "image_too_small"
        return rec
    out = p(f"{cfg['paths']['images']}/{row['id']}.jpg")
    im.save(out, "JPEG", quality=90)
    rec.update(status="ok", image_path=str(out.relative_to(p("."))), width=im.size[0], height=im.size[1])
    return rec


def summarise(cfg: dict) -> dict:
    d = cfg["paths"]["processed"]
    man = p(f"{d}/fnn_manifest.jsonl")
    rows = [json.loads(line) for line in open(man)] if man.exists() else []
    if not rows:
        print("[fnn] manifest empty")
        return {}
    df = pd.DataFrame(rows)
    ok = df[df["status"] == "ok"].copy()
    ok["label"] = ok["fnn_label"].map({"fake": "Fabricated", "real": "Credible"})
    ok[["id", "title", "fnn_label", "label", "site", "image_path", "news_url"]] \
        .to_csv(p(f"{d}/fnn_pairs.csv"), index=False)
    s = {
        "attempted": len(df),
        "usable_pairs": len(ok),
        "usable_rate": round(len(ok) / len(df), 3),
        "status_counts": df["status"].value_counts().to_dict(),
        "usable_by_label": ok["fnn_label"].value_counts().to_dict(),
        "usable_by_site": ok["site"].value_counts().to_dict(),
        "usable_via": ok["via"].value_counts().to_dict(),
    }
    write_json(s, f"{d}/fnn_image_summary.json")
    print(json.dumps(s, indent=2))
    return s


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=None, help="override sample_size")
    ap.add_argument("--summary-only", action="store_true")
    args = ap.parse_args()
    cfg = load_config()
    if args.summary_only:
        summarise(cfg)
        return

    df = load_fnn(p(cfg["fakenewsnet"]["dir"]))
    print(f"[fnn] {len(df)} articles with URL+title | "
          f"{df.groupby(['site','fnn_label']).size().to_dict()}")
    sample = sample_balanced(df, args.limit or cfg["fakenewsnet"]["sample_size"], cfg["seed"])

    man = p(f"{cfg['paths']['processed']}/fnn_manifest.jsonl")
    done = {json.loads(line)["id"] for line in open(man)} if man.exists() else set()
    todo = [r for r in sample.to_dict("records") if r["id"] not in done]
    print(f"[fnn] sample={len(sample)} already_done={len(done)} todo={len(todo)}")

    n = 0
    with ThreadPoolExecutor(cfg["fakenewsnet"]["workers"]) as ex, open(man, "a") as fout:
        futs = [ex.submit(process, r, cfg) for r in todo]
        for fut in as_completed(futs):
            rec = fut.result()
            with _lock:
                fout.write(json.dumps(rec) + "\n")
                fout.flush()
            n += 1
            if n % 50 == 0:
                print(f"[fnn] {n}/{len(todo)}", flush=True)
    summarise(cfg)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/06_train_baseline.py
"""Week-1 baseline: RoBERTa-base, five classes, class-weighted cross-entropy.

This is the "fine-tuned RoBERTa without the proposed modules" baseline from the
evaluation plan. It is also the victim model for TextFooler generation.

Input : data/processed/corpus_{train,valid,test}.csv, corpus_stats.json
Output: runs/baseline/best/            (model + tokenizer, chosen by valid macro-F1)
        runs/baseline/metrics.json     (valid + test: acc, macro-F1, per-class, per-source)
        runs/baseline/confusion_test.csv

Usage : python scripts/06_train_baseline.py [--epochs N] [--smoke]
"""
from __future__ import annotations

import argparse
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch import nn
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import class_names, load_config, p, set_seed, write_json  # noqa: E402


class TextDS(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, tok, max_len: int):
        self.enc = tok(df["text"].tolist(), truncation=True, max_length=max_len)
        self.y = df["label_id"].astype(int).tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item


class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights: torch.Tensor, **kw):
        super().__init__(*a, **kw)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.functional.cross_entropy(out.logits, labels,
                                           weight=self.class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss


def metrics_fn(eval_pred):
    logits, labels = eval_pred
    pred = logits.argmax(-1)
    present = sorted(set(labels.tolist()))
    return {"accuracy": accuracy_score(labels, pred),
            "macro_f1": f1_score(labels, pred, labels=present, average="macro", zero_division=0)}


def report(df: pd.DataFrame, pred: np.ndarray, classes: list[str]) -> dict:
    y = df["label_id"].to_numpy()
    present = sorted(set(y.tolist()))
    r = {
        "n": len(df),
        "accuracy": round(accuracy_score(y, pred), 4),
        "macro_f1": round(f1_score(y, pred, labels=present, average="macro", zero_division=0), 4),
        "classes_in_eval": [classes[i] for i in present],
        "per_class": classification_report(y, pred, labels=list(range(len(classes))),
                                           target_names=classes, output_dict=True, zero_division=0),
        "per_source": {},
    }
    for src, idx in df.groupby("source").groups.items():
        ys, ps = y[df.index.get_indexer(idx)], pred[df.index.get_indexer(idx)]
        pr = sorted(set(ys.tolist()))
        r["per_source"][src] = {"n": len(ys), "accuracy": round(accuracy_score(ys, ps), 4),
                                "macro_f1": round(f1_score(ys, ps, labels=pr, average="macro",
                                                           zero_division=0), 4)}
    return r


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=float, default=None)
    ap.add_argument("--smoke", action="store_true", help="tiny model + 200 rows, CPU-friendly check")
    ap.add_argument("--lr", type=float, default=None)
    ap.add_argument("--run-name", default="baseline", help="output folder under runs/")
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--out-dir", default=None, help="absolute output folder (e.g. Colab local disk instead of Drive)")
    args = ap.parse_args()

    cfg = load_config()
    if args.seed is not None:
        cfg["seed"] = args.seed
    bc = cfg["baseline"]
    set_seed(cfg["seed"])
    classes = class_names()
    d = cfg["paths"]["processed"]
    out_dir = Path(args.out_dir) if args.out_dir else p(f"{cfg['paths']['runs']}/{args.run_name}")
    out_dir.mkdir(parents=True, exist_ok=True)

    train, valid, test = (pd.read_csv(p(f"{d}/corpus_{s}.csv")) for s in ["train", "valid", "test"])
    stats = json.load(open(p(f"{d}/corpus_stats.json")))
    weights = torch.tensor([stats["class_weights"][c] for c in classes], dtype=torch.float)

    model_name = bc["model"]
    if args.smoke:
        model_name = "hf-internal-testing/tiny-random-RobertaForSequenceClassification"
        train, valid, test = train.head(200), valid.head(100), test.head(100)

    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(classes),
        id2label=dict(enumerate(classes)), label2id={c: i for i, c in enumerate(classes)},
        ignore_mismatched_sizes=args.smoke)

    ds = {k: TextDS(v, tok, bc["max_len"]) for k, v in
          [("train", train), ("valid", valid), ("test", test)]}

    targs = TrainingArguments(
        output_dir=str(out_dir / "ckpt"),
        num_train_epochs=args.epochs or bc["epochs"],
        per_device_train_batch_size=bc["batch_size"],
        per_device_eval_batch_size=bc["batch_size"] * 2,
        learning_rate=args.lr or bc["lr"],
        # warmup as an integer step count: works on transformers 4.x and 5.x
        # (5.x removed warmup_ratio)
        warmup_steps=int(math.ceil(len(train) / bc["batch_size"])
                         * (args.epochs or bc["epochs"]) * bc["warmup_ratio"]),
        weight_decay=bc["weight_decay"],
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=bc["fp16"] and torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        seed=cfg["seed"],
    )
    trainer = WeightedTrainer(model=model, args=targs, train_dataset=ds["train"],
                              eval_dataset=ds["valid"], data_collator=DataCollatorWithPadding(tok),
                              compute_metrics=metrics_fn, class_weights=weights)
    trainer.train()
    trainer.save_model(str(out_dir / "best"))
    tok.save_pretrained(str(out_dir / "best"))

    results = {"model": model_name, "class_weights": stats["class_weights"], "config": bc,
               "run_name": args.run_name, "seed": cfg["seed"], "lr": args.lr or bc["lr"], "epochs": args.epochs or bc["epochs"]}
    for name, df in [("valid", valid), ("test", test)]:
        pred = trainer.predict(ds[name]).predictions.argmax(-1)
        results[name] = report(df.reset_index(drop=True), pred, classes)
        print(f"[{args.run_name}] {name}: acc={results[name]['accuracy']} macro_f1={results[name]['macro_f1']} "
              f"per_source={results[name]['per_source']}")
        if name == "test":
            cm = confusion_matrix(df["label_id"], pred, labels=list(range(len(classes))))
            pd.DataFrame(cm, index=classes, columns=classes).to_csv(out_dir / "confusion_test.csv")
            pd.DataFrame({"id": df["id"], "text": df["text"], "label": df["label"],
                          "pred": [classes[i] for i in pred]}).to_csv(out_dir / "test_predictions.csv",
                                                                       index=False)
    write_json(results, out_dir / "metrics.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/07_textfooler_generate.py
"""Generate TextFooler adversarial examples against the week-1 baseline.

Uses src/xfakenet/textfooler.py (the Jin et al. algorithm, no TextAttack needed).

--split train : attacks TRAIN statements until the number of SUCCESSFUL examples
                reaches 15% of the final corpus (70/15/15), with per-class quotas.
                These are the augmentation data for week 3 (gold label kept).
--split test  : attacks a fixed stratified sample of the TEST split. The attack
                success rate is the robustness number for week 4. Never trained on.

Resumable: ids already in the output file are skipped and existing successes count.

Usage: python scripts/07_textfooler_generate.py --split train
       python scripts/07_textfooler_generate.py --split test
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import urllib.request
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed  # noqa: E402
from xfakenet import textfooler as tf  # noqa: E402

TF_VERSION = 2   # v2: names/numbers protected, antonyms + offensive words blocked


def synonym_source(cfg: dict):
    import zipfile
    tc = cfg["textfooler"]
    path = p(tc["embeddings_path"])
    urls = [tc["embeddings_url"],
            "https://media.githubusercontent.com/media/nmrksic/counter-fitting/master/word_vectors/counter-fitted-vectors.txt.zip"]
    for url in urls:
        if path.exists() and zipfile.is_zipfile(path):
            break
        if path.exists():
            os.remove(path)
        print(f"[textfooler] downloading counter-fitted vectors from {url} ...", flush=True)
        try:
            urllib.request.urlretrieve(url, path)
        except Exception as e:
            print(f"[textfooler] download failed ({e})", flush=True)
    if path.exists() and not zipfile.is_zipfile(path):
        os.remove(path)
        print("[textfooler] download was not a valid zip; using WordNet synonyms instead", flush=True)
    if path.exists():
        print("[textfooler] synonyms: counter-fitted vectors", flush=True)
        return tf.CounterFitted(str(path), k=tc["k"], min_cos=0.5), "counter-fitted"
    import nltk
    nltk.download("wordnet", quiet=True)
    return tf.WordNetSyn(k=tc["k"]), "wordnet"


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="train", choices=["train", "test"])
    ap.add_argument("--n", type=int, default=None)
    ap.add_argument("--victim", default=None)
    ap.add_argument("--victim-kind", default="hf", choices=["hf", "xfakenet"],
                    help="hf = Hugging Face classifier (baseline); xfakenet = XFakeNet model (week 3)")
    ap.add_argument("--out", default=None, help="override output file")
    args = ap.parse_args()

    cfg = load_config()
    tc = cfg["textfooler"]
    set_seed(cfg["seed"])

    import nltk
    for r in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger", "wordnet"]:
        nltk.download(r, quiet=True)

    syn, syn_name = synonym_source(cfg)
    victim = str(p(args.victim or tc["victim_dir"]))
    if args.victim_kind == "xfakenet":
        from xfakenet.model import Predictor
        predict_fn = Predictor(victim).predict_proba
    else:
        predict_fn = tf.make_predict(victim, cfg["baseline"]["max_len"])
    attacker = tf.TextFooler(
        predict=predict_fn,
        neighbours=syn, similarity=tf.make_similarity(tc["sim_model"]), pos_ok=tf.make_pos_ok(),
        protect=tf.make_protect(), banned=tf.make_banned(),
        sim_threshold=tc["sim_threshold"], max_change=tc["max_change"], query_budget=tc["query_budget"])

    df = pd.read_csv(p(f"{cfg['paths']['processed']}/corpus_{args.split}.csv"))
    df["id"] = df["id"].astype(str)
    out = p(args.out) if args.out else p(tc["out_train"] if args.split == "train" else tc["out_test"])
    prev = [json.loads(line) for line in open(out)] if out.exists() else []
    if prev and prev[0].get("tf_version") != TF_VERSION:
        old = out.with_name(out.stem + ".v1_unfiltered.jsonl")
        out.rename(old)
        print(f"[textfooler] archived old unfiltered output to {old.name}; regenerating", flush=True)
        prev = []
    done = {r["id"] for r in prev}

    if args.split == "train":
        a = cfg["augmentation"]
        target = args.n or round(len(df) * a["textfooler_share"] / a["original_share"])
        frac = df["label"].value_counts(normalize=True)
        quota = {c: round(target * f) for c, f in frac.items()}
        have = pd.Series([r["label"] for r in prev if r["status"] == "success"]).value_counts().to_dict()
        order = df.sample(frac=1.0, random_state=cfg["seed"])        # whole train, shuffled
    else:
        target = args.n or tc["test_n"]
        frac = df["label"].value_counts(normalize=True)
        order = pd.concat([g.sample(min(len(g), round(target * frac[c])), random_state=cfg["seed"])
                           for c, g in df.groupby("label")]).sample(frac=1.0, random_state=cfg["seed"])
        quota, have = None, {}
    todo = order[~order["id"].isin(done)]
    print(f"[textfooler] victim={victim} split={args.split} synonyms={syn_name} target={target} "
          f"already_done={len(done)} successes_so_far={sum(have.values())}", flush=True)

    t0, n, counts = time.time(), 0, {}
    with open(out, "a") as f:
        for row in todo.itertuples(index=False):
            if quota is not None:
                if all(have.get(c, 0) >= q for c, q in quota.items()):
                    break
                if have.get(row.label, 0) >= quota[row.label]:
                    continue
            try:
                res = attacker.attack(row.text, int(row.label_id))
            except Exception as e:  # keep the long job alive
                res = {"status": "error", "error": repr(e)[:200]}
            rec = {"tf_version": TF_VERSION, "id": row.id, "label": row.label, "label_id": int(row.label_id),
                   "source": row.source, "original": row.text, "synonyms": syn_name, **res}
            f.write(json.dumps(rec) + "\n")
            f.flush()
            counts[res["status"]] = counts.get(res["status"], 0) + 1
            if res["status"] == "success":
                have[row.label] = have.get(row.label, 0) + 1
            n += 1
            if n % 50 == 0:
                rate = n / (time.time() - t0)
                print(f"[textfooler] {n} attacked | {counts} | successes {sum(have.values())}/{target} "
                      f"| {rate:.1f}/s", flush=True)
    print(f"[textfooler] finished {counts} | successes by class {have}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile docs/label_mapping.md
# Label mapping (Week 1)

**Status:** fixed 20 Oct 2026. Source of truth: `configs/label_mapping.yaml`.

## Output classes

| id | Class | Meaning |
|---|---|---|
| 0 | Fabricated | Claim is false; no factual basis |
| 1 | Misleading | Contains truth but distorts, omits context or exaggerates |
| 2 | Credible | Accurate or substantially accurate |

## LIAR → XFakeNet

| LIAR rating | PolitiFact definition (short) | XFakeNet |
|---|---|---|
| pants-fire | false and ridiculous | Fabricated |
| false | not accurate | Fabricated |
| barely-true | element of truth, ignores critical facts | Misleading |
| half-true | partially accurate, leaves out details | Misleading |
| mostly-true | accurate but needs clarification | Credible |
| true | accurate, nothing significant missing | Credible |

**Rationale.** The split follows PolitiFact's own definitions: the two ratings
with no factual basis become Fabricated, the two that contain truth but
distort it become Misleading, and the two judged accurate become Credible.

**Known limitation.** Neighbouring LIAR ratings (e.g. false vs barely-true) are
hard to separate even for humans, so the class boundaries are noisy.

## Why Satire and Propaganda were dropped

The original plan had five classes. LIAR has no Satire or Propaganda examples,
so these were taken from TSHP-17 (Rashkin et al., 2017). In the first baseline
run, Propaganda scored F1 0.99 on validation but 0.08 on test. The TSHP-17 test
set uses different news websites from training, so the model had learned each
website's writing style rather than propaganda itself; most test propaganda
was labelled Satire. With no reliable source for these two classes in the
time available, the classifier uses three classes.

| Run | Accuracy | Macro-F1 | Note |
|---|---|---|---|
| Five-class (LIAR + TSHP-17) | 0.482 | 0.434 | Propaganda F1 0.08 on test |
| Three-class (LIAR only) | 0.482 | 0.470 | Adopted |

## FakeNewsNet

Binary fake/real → Fabricated/Credible. Used for C-score calibration and
evaluation only, not for classifier training.

## 3. Install packages

In [ ]:
!pip install -q gdown pyyaml beautifulsoup4
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA", torch.cuda.is_available())

## 4. Download the datasets
**LIAR** (official link; if it fails you'll get an upload box), **TSHP-17** for Satire/Propaganda, **FakeNewsNet** CSVs.

In [ ]:
import os, zipfile, glob, shutil
os.makedirs("data/raw/liar", exist_ok=True)
if not os.path.exists("data/raw/liar/train.tsv"):
    !wget -q -O /content/liar_dataset.zip https://www.cs.ucsb.edu/~william/data/liar_dataset.zip
    ok = os.path.exists("/content/liar_dataset.zip") and zipfile.is_zipfile("/content/liar_dataset.zip")
    if not ok:
        print("Official LIAR link failed. Download liar_dataset.zip (e.g. search 'LIAR dataset' on Kaggle) and upload it:")
        from google.colab import files
        up = files.upload()
        name = next(iter(up))
        shutil.move(name, "/content/liar_dataset.zip")
    with zipfile.ZipFile("/content/liar_dataset.zip") as z:
        z.extractall("data/raw/liar")
    for f in glob.glob("data/raw/liar/**/*.tsv", recursive=True):   # flatten if zipped inside a folder
        shutil.move(f, "data/raw/liar/" + os.path.basename(f)) if os.path.dirname(f) != "data/raw/liar" else None
print("LIAR files:", sorted(os.listdir("data/raw/liar")))

In [ ]:
# TSHP-17 (Rashkin et al. 2017) – Satire + Propaganda source.
# The original Google Drive link is dead; this is the CompareNet (ACL 2021) re-upload on GitHub.
import os, glob, shutil, zipfile
T = "data/raw/tshp17"; os.makedirs(T, exist_ok=True)
if not os.path.exists(f"{T}/fulltrain.csv"):
    !wget -q -O /content/raw_data.zip https://github.com/BUPT-GAMMA/CompareNet_FakeNewsDetection/releases/download/dataset/raw_data.zip
    zipfile.ZipFile("/content/raw_data.zip").extractall("/content/tshp_x")
    for name in ["fulltrain.csv", "balancedtest.csv"]:
        hits = glob.glob(f"/content/tshp_x/**/{name}", recursive=True)
        if hits: shutil.copy(hits[0], f"{T}/{name}")
print("TSHP-17 files:", sorted(os.listdir(T)))

In [ ]:
if not os.path.exists("data/raw/FakeNewsNet"):
    !git clone -q --depth 1 https://github.com/KaiDMML/FakeNewsNet.git data/raw/FakeNewsNet
print("FakeNewsNet CSVs:", sorted(os.listdir("data/raw/FakeNewsNet/dataset")))

## 5. Clean the data and apply the three-class mapping

In [ ]:
!python scripts/02_prepare_liar.py
!python scripts/03_prepare_extra_classes.py
!python scripts/04_build_corpus.py

## 6. Start the FakeNewsNet image download (runs in the background)
It tries 1,000 articles and counts how many still have a usable headline + image. Takes 30–90 min; training continues meanwhile.

In [ ]:
import subprocess
img_proc = subprocess.Popen("python scripts/05_fetch_fnn_images.py > logs/fnn_images.log 2>&1", shell=True)
print("Image download started in background. Progress: logs/fnn_images.log")

## 7. Train the baseline (plain RoBERTa, full dataset, ~10 min on T4)

In [ ]:
!python scripts/06_train_baseline.py 2>&1 | grep -v "it/s\]" | tail -40

## 8. Results for your mentor
Re-run this cell later if the image download was still running.

In [ ]:
import json, os, pandas as pd, shutil
m = json.load(open("runs/baseline/metrics.json"))
cs = json.load(open("data/processed/corpus_stats.json"))
import yaml
classes = yaml.safe_load(open("configs/label_mapping.yaml"))["classes"]

lines = ["# XFakeNet – Week 1 results", "",
         f"## Dataset ({len(classes)}-class corpus, LIAR)", "",
         "| Split | " + " | ".join(classes) + " | Total |", "|---" * (len(classes) + 2) + "|"]
for s in ["train", "valid", "test"]:
    c = cs[s]["counts"]
    lines.append(f"| {s} | " + " | ".join(str(c[k]) for k in classes) + f" | {cs[s]['n']} |")

t = m["test"]
lines += ["", "## Baseline: RoBERTa-base, class-weighted cross-entropy (test set)", "",
          f"- **Accuracy:** {t['accuracy']:.3f}", f"- **Macro-F1:** {t['macro_f1']:.3f}", "",
          "| Class | Precision | Recall | F1 | Support |", "|---|---|---|---|---|"]
for k in classes:
    r = t["per_class"][k]
    lines.append(f"| {k} | {r['precision']:.3f} | {r['recall']:.3f} | {r['f1-score']:.3f} | {int(r['support'])} |")

f = "data/processed/fnn_image_summary.json"
running = 'img_proc' in globals() and img_proc.poll() is None
if running:
    !python scripts/05_fetch_fnn_images.py --summary-only > /dev/null 2>&1
if os.path.exists(f):
    s = json.load(open(f))
    lines += ["", "## FakeNewsNet image-text pairs" + (" (download still running – partial)" if running else ""), "",
              f"- Articles tried: {s['attempted']}", f"- **Usable headline+image pairs: {s['usable_pairs']}** "
              f"({s['usable_rate']*100:.0f}%)", f"- By label: {s['usable_by_label']}",
              f"- Failure reasons: {s['status_counts']}"]

report = "\n".join(lines)
os.makedirs("results", exist_ok=True)
open("results/week1_summary.md", "w").write(report)
from IPython.display import Markdown, display
display(Markdown(report))

for p in ["runs/baseline/metrics.json", "runs/baseline/confusion_test.csv", "data/processed/corpus_stats.json",
          "data/processed/fnn_image_summary.json", "docs/label_mapping.md"]:
    if os.path.exists(p): shutil.copy(p, "results/")
shutil.make_archive("/content/xfakenet_week1_results", "zip", "results")
from google.colab import files
files.download("/content/xfakenet_week1_results.zip")

## 9. (Optional) TextFooler adversarial examples

Takes **many hours** for 2,500 examples. It saves after every example to Drive, so if Colab disconnects
just run this cell again and it continues where it stopped. Run this **after** section 8, since installing
TextAttack can change package versions.

In [ ]:
!pip install -q "textattack[tensorflow]"
import subprocess
tf_proc = subprocess.Popen("python scripts/07_textfooler_generate.py > logs/textfooler.log 2>&1", shell=True)
print("TextFooler started. Check progress with the next cell.")

In [ ]:
!tail -5 logs/textfooler.log